# Notebook 10 — Offline distillation recovery on Taylor 20% pruned Qwen 2.5 3B

**The hypothesis.** Notebook 5 showed that LoRA fine-tuning on narrow MMLU-aux data didn't recover pruning damage — it overwrote one answer-format prior with another. Notebook 9 confirmed the real bottleneck is not capacity (rank doesn't help) but the training signal: narrow MCQ data can't repair MLP circuits trained on the pretraining distribution.

**The cleaner approach: offline knowledge distillation.** Use the unpruned dense Qwen 2.5 3B as teacher. Cache its top-K=64 logits at every relevant token position (done in notebook 9, ~3.3 MB on disk). Train a Taylor 20% pruned + LoRA student to match the teacher's output distribution at those positions via KL divergence. The training signal now contains the teacher's *full* output preference, not just answer-format priors.

**Why 20% pruning** (not 10%): user's call. Tests whether distillation can recover from heavier damage. At 20% MMLU was already at random floor (26.78%); on CoNLL F1 we expect a bigger gap than the 10% baseline (0.383). Recovery from a worse starting point is a stronger demonstration if it works.

## Pipeline

1. Load Qwen 2.5 3B, cache MLP weights, compute Taylor importance.
2. Apply Taylor 20% mask (in-place).
3. Eval pre-distillation: perplexity + F1 (the new 20% baseline).
4. Load teacher cache from `results/teacher_conll_logits.pt`.
5. Attach LoRA (r=16, full-precision AdamW — the only validated config).
6. Distillation training: KL divergence between student logits at teacher's top-K positions vs teacher's softmaxed top-K logits. Combined with hard CE on gold target tokens.
7. Eval post-distillation: perplexity + F1.
8. Compare: how much of the gap from pre-distillation to dense (F1=0.614) closes?

Train/eval split: F1 eval samples are drawn from `setdiff1d(all_indices, ppl_indices)` so there is **zero overlap** between training (teacher cache) and F1 eval. Honest comparison.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, json, math, re, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import config

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TILE_R, TILE_C = config.TILE_SIZES[0]
PRUNE_SPARSITY = 0.20  # user's call — distil from 20% pruned

TEACHER_CACHE_PATH = "results/teacher_conll_logits.pt"
N_F1_SAMPLES = 200
MAX_NEW_TOKENS = 64

# Distillation hyperparams
DISTILL_T = 2.0       # softmax temperature (Hinton)
DISTILL_ALPHA = 0.5   # mixing weight: alpha * KL + (1-alpha) * CE_hard
EPOCHS = 3
LR = 1e-4
MAX_TRAIN_LEN = 320   # CoNLL prompts (with few-shot prefix) are ~250 tokens

torch.set_float32_matmul_precision("high")
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

GPU free: 7.53 / 8.18 GB


## 1. Load model + cache MLP weights

In [2]:
print(f"loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()
model.config.use_cache = True

def is_mlp_name(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

ORIGINAL_MLP_WEIGHTS = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and is_mlp_name(name):
        ORIGINAL_MLP_WEIGHTS[name + ".weight"] = module.weight.data.detach().clone().cpu()

print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"cached {len(ORIGINAL_MLP_WEIGHTS)} MLP matrices on CPU")

loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

params: 3.09B
cached 108 MLP matrices on CPU


## 2. Load CoNLL-2003 dev + prompt format (matching notebook 9)

In [3]:
ds = load_dataset("eriktks/conll2003", split="validation", trust_remote_code=True)
TAG_ID_TO_NAME = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG",
                  5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}

def entities_from_bio(tokens, tag_ids):
    entities = []
    cur_words, cur_type = None, None
    for tok, tid in zip(tokens, tag_ids):
        tag = TAG_ID_TO_NAME[tid]
        if tag.startswith("B-"):
            if cur_words:
                entities.append((" ".join(cur_words), cur_type))
            cur_words = [tok]; cur_type = tag[2:]
        elif tag.startswith("I-") and cur_words and cur_type == tag[2:]:
            cur_words.append(tok)
        else:
            if cur_words:
                entities.append((" ".join(cur_words), cur_type))
                cur_words = None; cur_type = None
    if cur_words:
        entities.append((" ".join(cur_words), cur_type))
    return entities

def format_entities(entities):
    return "none" if not entities else ", ".join(f"{txt} ({typ})" for txt, typ in entities)

FEW_SHOT = [
    ("Angela Merkel visited Berlin last Friday .", [("Angela Merkel","PER"),("Berlin","LOC")]),
    ("Apple Inc. acquired Beats Electronics in 2014 .", [("Apple Inc.","ORG"),("Beats Electronics","ORG")]),
    ("The Olympic Games will return to Paris in 2024 .", [("Olympic Games","MISC"),("Paris","LOC")]),
]
FEW_SHOT_PREFIX = "Extract named entities from each text. Answer with comma-separated 'entity (TYPE)' pairs, where TYPE is PER, ORG, LOC, or MISC. If there are no entities, write 'none'.\n\n"
for text, ents in FEW_SHOT:
    FEW_SHOT_PREFIX += f"Text: {text}\nEntities: {format_entities(ents)}\n\n"

def make_prompt(tokens):
    return FEW_SHOT_PREFIX + f"Text: {' '.join(tokens)}\nEntities:"
def make_full(tokens, entities):
    return make_prompt(tokens) + " " + format_entities(entities)

print(f"CoNLL dev: {len(ds)} sentences")

CoNLL dev: 3250 sentences


## 3. Disjoint train/eval splits — train on teacher cache indices, eval on the rest

In [4]:
teacher_data = torch.load(TEACHER_CACHE_PATH, weights_only=False)
ppl_indices_used = teacher_data["ppl_indices"]   # 500 indices used for training
teacher_cache = teacher_data["cache"]            # dict[i in 0..N-1] -> tensors (NOT keyed by ds index)

print(f"loaded teacher cache: {len(teacher_cache)} examples, top-K={teacher_data['k']}")
print(f"dense baselines from cache: PPL={teacher_data['dense_ppl']:.3f}, F1={teacher_data['dense_f1']:.3f}")

# Pick F1 eval indices DISJOINT from training set
rng = np.random.RandomState(43)
all_indices = np.arange(len(ds))
remaining = np.setdiff1d(all_indices, np.array(ppl_indices_used))
f1_eval_indices = rng.choice(remaining, size=N_F1_SAMPLES, replace=False)
print(f"F1 eval: {len(f1_eval_indices)} sentences, disjoint from {len(ppl_indices_used)} training samples")

loaded teacher cache: 500 examples, top-K=64
dense baselines from cache: PPL=1.725, F1=0.614
F1 eval: 200 sentences, disjoint from 500 training samples


## 4. C4 calibration data + Taylor importance + masking utilities (reused from nb9)

In [5]:
N_CAL = 512
SEQ_LEN_CAL = 128

cal_ds = load_dataset(config.CALIBRATION_DATASET, config.CALIBRATION_SUBSET,
                     split="train", streaming=True, trust_remote_code=True)
cal_texts = []
for ex in cal_ds:
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
    if len(cal_texts) >= N_CAL:
        break
cal_encodings = [tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_CAL, truncation=True)
                 for t in cal_texts]
print(f"calibration samples: {len(cal_encodings)}")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

calibration samples: 512


In [6]:
def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def compute_taylor_importance(model, cal_encodings, tag=""):
    for p in model.parameters():
        p.requires_grad_(False)
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2:
            p.requires_grad_(True)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    importance_gpu = {}
    def make_hook(pname):
        def hook(param):
            if param.grad is None: return
            taylor = (param.data * param.grad).abs().float()
            out_dim, in_dim = taylor.shape
            nr, nc = out_dim // TILE_R, in_dim // TILE_C
            tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
            tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
            if pname in importance_gpu: importance_gpu[pname] += tile_scores
            else: importance_gpu[pname] = tile_scores.clone()
            param.grad = None
        return hook

    handles = []
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2 and p.requires_grad:
            handles.append(p.register_post_accumulate_grad_hook(make_hook(name)))

    n_samples = 0
    for enc in tqdm(cal_encodings, desc=f"taylor[{tag}]"):
        model.zero_grad(set_to_none=True)
        inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
        out = model(**inputs, labels=inputs["input_ids"])
        out.loss.backward()
        n_samples += 1
    for h in handles: h.remove()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    for p in model.parameters(): p.requires_grad_(False)
    importance = {name: (v / n_samples).cpu().numpy() for name, v in importance_gpu.items()}
    del importance_gpu; gc.collect(); torch.cuda.empty_cache()
    return importance

def apply_taylor_masks_at(importance, sparsity):
    comp_stats = {}
    for ct in config.PRUNE_TARGETS_PATTERNS:
        vals = np.concatenate([m.ravel() for n, m in importance.items() if ct in n])
        comp_stats[ct] = (vals.mean(), vals.std())
    norm = {}
    for name, m in importance.items():
        mu, sd = comp_stats[get_component_type(name)]
        norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)
    masks = {}
    for name, m in norm.items():
        thr = float(np.percentile(m.ravel(), sparsity * 100))
        masks[name] = m < thr
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in masks:
                    mask = masks[pn]
                    w = module.weight.data
                    out_f, in_f = w.shape
                    nr, nc = out_f // TILE_R, in_f // TILE_C
                    for i in range(nr):
                        for j in range(nc):
                            if mask[i, j]:
                                w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
    return masks

def restore_mlp_weights():
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in ORIGINAL_MLP_WEIGHTS:
                    module.weight.data.copy_(ORIGINAL_MLP_WEIGHTS[pn].to(module.weight.device))

## 5. Eval functions (perplexity + F1, matching notebook 9)

In [7]:
def parse_entities_from_completion(text):
    text = text.strip().split("\n")[0].strip()
    if not text or text.lower().startswith("none"):
        return []
    out = []
    for part in text.split(","):
        m = re.match(r"^(.+?)\s*\(([A-Z]+)\)\s*$", part.strip())
        if m:
            out.append((m.group(1).strip(), m.group(2).strip()))
    return out

def micro_f1(preds, refs):
    tp = fp = fn = 0
    for p, r in zip(preds, refs):
        ps, rs = set(p), set(r)
        tp += len(ps & rs); fp += len(ps - rs); fn += len(rs - ps)
    P = tp/(tp+fp) if tp+fp else 0.0
    R = tp/(tp+fn) if tp+fn else 0.0
    F = 2*P*R/(P+R) if P+R else 0.0
    return P, R, F, tp, fp, fn

def eval_perplexity(model, examples, tag=""):
    total_nll, total_tokens = 0.0, 0
    for ex in tqdm(examples, desc=f"ppl[{tag}]"):
        toks = ex["tokens"]
        ents = entities_from_bio(toks, ex["ner_tags"])
        prompt = make_prompt(toks); full = make_full(toks, ents)
        prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0]
        full_ids = tokenizer(full, return_tensors="pt")["input_ids"][0]
        if len(full_ids) <= len(prompt_ids):
            continue
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(config.DEVICE)).logits[0]
        ans_pred_start = len(prompt_ids) - 1
        ans_pred_end = len(full_ids) - 1
        target_ids = full_ids[len(prompt_ids):].to(config.DEVICE)
        ans_logits = logits[ans_pred_start:ans_pred_end]
        log_probs = F.log_softmax(ans_logits.float(), dim=-1)
        nlls = -log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
        total_nll += nlls.sum().item(); total_tokens += nlls.numel()
    avg_nll = total_nll / max(total_tokens, 1)
    return {"perplexity": math.exp(avg_nll), "nll": avg_nll, "tokens": total_tokens}

def eval_f1(model, examples, max_new=MAX_NEW_TOKENS, tag=""):
    preds, refs = [], []
    for ex in tqdm(examples, desc=f"F1[{tag}]"):
        toks = ex["tokens"]
        refs.append(entities_from_bio(toks, ex["ner_tags"]))
        prompt = make_prompt(toks)
        enc = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(
                input_ids=enc["input_ids"].to(config.DEVICE),
                attention_mask=enc["attention_mask"].to(config.DEVICE),
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        preds.append(parse_entities_from_completion(completion))
    P, R, Fs, tp, fp, fn = micro_f1(preds, refs)
    return {"precision": P, "recall": R, "f1": Fs, "tp": tp, "fp": fp, "fn": fn,
            "sample_predictions": preds[:5], "sample_references": refs[:5]}

## 6. Calibrate Taylor + apply 20% mask + measure pre-distillation baseline

In [8]:
print("Taylor calibration on dense model...")
importance_taylor = compute_taylor_importance(model, cal_encodings, tag="dense")
print(f"calibrated, peak GPU {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

model.eval(); model.config.use_cache = True
restore_mlp_weights()
masks = apply_taylor_masks_at(importance_taylor, PRUNE_SPARSITY)
actual = sum(m.sum() for m in masks.values()) / sum(m.size for m in masks.values())
print(f"applied Taylor {int(PRUNE_SPARSITY*100)}% mask  (actual sparsity {actual*100:.1f}%)")

Taylor calibration on dense model...


taylor[dense]: 100%|███████████████████████████████████████████████| 512/512 [03:20<00:00,  2.56it/s]


calibrated, peak GPU 7.01 GB
applied Taylor 20% mask  (actual sparsity 20.0%)


In [9]:
f1_eval_examples = [ds[int(i)] for i in f1_eval_indices]

print("=== Pre-distillation baseline (Taylor 20% pruned, no LoRA) ===")
t0 = time.time()
ppl_pre = eval_perplexity(model, f1_eval_examples, tag="pre")
print(f"  PPL={ppl_pre['perplexity']:.3f}  NLL={ppl_pre['nll']:.3f}  [{time.time()-t0:.1f}s]")
t0 = time.time()
f1_pre = eval_f1(model, f1_eval_examples, tag="pre")
print(f"  F1={f1_pre['f1']:.3f}  P={f1_pre['precision']:.3f}  R={f1_pre['recall']:.3f}  [{time.time()-t0:.1f}s]")

=== Pre-distillation baseline (Taylor 20% pruned, no LoRA) ===


ppl[pre]: 100%|████████████████████████████████████████████████████| 200/200 [00:10<00:00, 19.84it/s]


  PPL=5.789  NLL=1.756  [10.1s]


F1[pre]: 100%|█████████████████████████████████████████████████████| 200/200 [05:00<00:00,  1.50s/it]

  F1=0.097  P=0.099  R=0.095  [300.4s]


## 7. Attach LoRA (r=16, full-precision AdamW — the only validated config)

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f"peak GPU after LoRA: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

trainable params: 22,560,768 || all params: 3,108,499,456 || trainable%: 0.7258
peak GPU after LoRA: 7.01 GB


## 8. Distillation loss + training loop

**Loss** at each answer position p:
- Teacher distribution = softmax(teacher_topk_logits[p] / T) over the K vocab indices the teacher cached.
- Student distribution = log-softmax(student_full_logits[p] / T) over **full vocab**, gathered at those K indices.
- KL(teacher ‖ student) = Σ_k teacher_p[k] · (log teacher_p[k] − student_log_p_at_k[k]).
- Multiply by T² (Hinton 2015).
- Hard CE on the gold target token at position p.
- Final: `α · KL_distill + (1−α) · CE_hard` summed across all answer positions in the sequence.

In [11]:
def distill_step_loss(student_logits, teacher_topk_idx, teacher_topk_logits,
                      target_ids, ans_pred_start, ans_pred_end, T, alpha):
    """Compute combined distillation + hard-CE loss for one example.
    student_logits: (seq, vocab) full student output
    teacher_topk_idx: (n_ans, K) cached teacher top-K indices at the answer positions
    teacher_topk_logits: (n_ans, K) teacher's logits at those K indices (bf16 cached → upcast)
    target_ids: (n_ans,) gold target tokens (one per answer position)
    ans_pred_start, ans_pred_end: range of POSITIONS in student_logits that predict answer tokens.
    """
    student_ans = student_logits[ans_pred_start:ans_pred_end].float()  # (n_ans, vocab)

    # Hard CE
    log_p_full = F.log_softmax(student_ans, dim=-1)
    ce_per_pos = -log_p_full.gather(1, target_ids.unsqueeze(1)).squeeze(1)  # (n_ans,)
    loss_ce = ce_per_pos.mean()

    # KL distillation with temperature
    teacher_p = F.softmax(teacher_topk_logits.float() / T, dim=-1)        # (n_ans, K)
    teacher_log_p = F.log_softmax(teacher_topk_logits.float() / T, dim=-1)
    student_log_p_T = F.log_softmax(student_ans / T, dim=-1)               # full vocab
    student_log_p_at_k = student_log_p_T.gather(1, teacher_topk_idx.long())  # (n_ans, K)
    kl_per_pos = (teacher_p * (teacher_log_p - student_log_p_at_k)).sum(dim=-1)  # (n_ans,)
    loss_kl = kl_per_pos.mean() * (T * T)

    return alpha * loss_kl + (1.0 - alpha) * loss_ce, loss_ce.detach(), loss_kl.detach()

# Build training batches from cache. Each batch is a single example (bs=1) for simplicity.
train_examples = []
skipped_long = 0
for cache_key, c in teacher_cache.items():
    if len(c["input_ids"]) > MAX_TRAIN_LEN:
        skipped_long += 1
        continue
    train_examples.append({
        "input_ids":     c["input_ids"],
        "prompt_len":    c["prompt_len"],
        "topk_indices":  c["topk_indices"].long(),
        "topk_logits":   c["topk_logits"].float(),
        "target_ids":    c["target_ids"],
    })
print(f"prepared {len(train_examples)} training examples (skipped {skipped_long} > {MAX_TRAIN_LEN} tokens)")

prepared 499 training examples (skipped 1 > 320 tokens)


In [12]:
model.train()
model.gradient_checkpointing_enable()
model.config.use_cache = False

optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

gc.collect(); torch.cuda.empty_cache()
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

losses_ce, losses_kl, losses_total = [], [], []
t0 = time.time()
for epoch in range(EPOCHS):
    order = np.random.RandomState(42 + epoch).permutation(len(train_examples))
    pbar = tqdm(order, desc=f"epoch {epoch+1}/{EPOCHS}")
    for idx in pbar:
        ex = train_examples[idx]
        input_ids = ex["input_ids"].unsqueeze(0).to(config.DEVICE)
        prompt_len = ex["prompt_len"]
        topk_idx   = ex["topk_indices"].to(config.DEVICE)
        topk_log   = ex["topk_logits"].to(config.DEVICE)
        target_ids = ex["target_ids"].to(config.DEVICE)

        out = model(input_ids=input_ids)
        student_logits = out.logits[0]
        ans_pred_start = prompt_len - 1
        ans_pred_end   = input_ids.shape[1] - 1
        n_ans = ans_pred_end - ans_pred_start
        if n_ans <= 0 or n_ans != topk_idx.shape[0]:
            continue  # safety guard

        loss, loss_ce, loss_kl = distill_step_loss(
            student_logits, topk_idx, topk_log, target_ids,
            ans_pred_start, ans_pred_end, T=DISTILL_T, alpha=DISTILL_ALPHA,
        )
        loss.backward()
        optim.step(); optim.zero_grad(set_to_none=True)
        losses_total.append(loss.item())
        losses_ce.append(loss_ce.item())
        losses_kl.append(loss_kl.item())
        if len(losses_total) % 50 == 0:
            pbar.set_postfix(L=f"{np.mean(losses_total[-50:]):.3f}",
                              CE=f"{np.mean(losses_ce[-50:]):.3f}",
                              KL=f"{np.mean(losses_kl[-50:]):.3f}")

print(f"\ntrained {len(losses_total)} steps in {time.time()-t0:.1f}s")
print(f"first 50: total={np.mean(losses_total[:50]):.3f}  CE={np.mean(losses_ce[:50]):.3f}  KL={np.mean(losses_kl[:50]):.3f}")
print(f"last 50:  total={np.mean(losses_total[-50:]):.3f}  CE={np.mean(losses_ce[-50:]):.3f}  KL={np.mean(losses_kl[-50:]):.3f}")
print(f"peak GPU during training: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

VRAM before training: 6.28 GB


epoch 3/3: 100%|██████████████████████| 499/499 [01:36<00:00,  5.19it/s, CE=0.291, KL=0.724, L=0.508]


trained 1497 steps in 288.1s
first 50: total=1.936  CE=1.170  KL=2.702
last 50:  total=0.407  CE=0.208  KL=0.606
peak GPU during training: 7.01 GB


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
              (k_proj): Linear(in_features=2048, out_features=256, bias=True)
              (v_proj): Linear(in_features=2048, out_features=256, bias=True)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): Qwen2MLP(
              (gate_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=11008, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
      

## 9. Post-distillation eval

In [13]:
print("=== Post-distillation eval (Taylor 20% pruned + LoRA distilled) ===")
t0 = time.time()
ppl_post = eval_perplexity(model, f1_eval_examples, tag="post")
print(f"  PPL={ppl_post['perplexity']:.3f}  NLL={ppl_post['nll']:.3f}  [{time.time()-t0:.1f}s]")
t0 = time.time()
f1_post = eval_f1(model, f1_eval_examples, tag="post")
print(f"  F1={f1_post['f1']:.3f}  P={f1_post['precision']:.3f}  R={f1_post['recall']:.3f}  [{time.time()-t0:.1f}s]")

=== Post-distillation eval (Taylor 20% pruned + LoRA distilled) ===


ppl[post]: 100%|███████████████████████████████████████████████████| 200/200 [00:12<00:00, 16.54it/s]


  PPL=1.406  NLL=0.341  [12.1s]


F1[post]: 100%|████████████████████████████████████████████████████| 200/200 [04:08<00:00,  1.24s/it]

  F1=0.423  P=0.632  R=0.318  [248.1s]


## 10. Summary — recovery rate

In [14]:
dense_ppl = teacher_data["dense_ppl"]
dense_f1  = teacher_data["dense_f1"]

ppl_gap_pre  = ppl_pre["perplexity"] - dense_ppl
ppl_gap_post = ppl_post["perplexity"] - dense_ppl
f1_gap_pre  = dense_f1 - f1_pre["f1"]
f1_gap_post = dense_f1 - f1_post["f1"]

ppl_recovery = (ppl_gap_pre - ppl_gap_post) / ppl_gap_pre * 100 if ppl_gap_pre > 0 else 0
f1_recovery  = (f1_gap_pre - f1_gap_post) / f1_gap_pre * 100 if f1_gap_pre > 0 else 0

print("=" * 72)
print(f"  Distillation recovery — Taylor 20% pruned Qwen 2.5 3B")
print("=" * 72)
print(f"  {'Metric':<22s}  {'Dense':>10s}  {'Pre-dist':>10s}  {'Post-dist':>10s}  {'Recovered':>12s}")
print("-" * 72)
print(f"  {'Perplexity':<22s}  {dense_ppl:>10.3f}  {ppl_pre['perplexity']:>10.3f}  {ppl_post['perplexity']:>10.3f}  {ppl_recovery:>10.1f}%")
print(f"  {'F1':<22s}  {dense_f1:>10.3f}  {f1_pre['f1']:>10.3f}  {f1_post['f1']:>10.3f}  {f1_recovery:>10.1f}%")
print(f"  {'Precision':<22s}  {'?':>10s}  {f1_pre['precision']:>10.3f}  {f1_post['precision']:>10.3f}")
print(f"  {'Recall':<22s}  {'?':>10s}  {f1_pre['recall']:>10.3f}  {f1_post['recall']:>10.3f}")
print("=" * 72)

results = {
    "model": MODEL_NAME, "prune_sparsity": PRUNE_SPARSITY,
    "distill": {"alpha": DISTILL_ALPHA, "T": DISTILL_T, "epochs": EPOCHS, "lr": LR},
    "dense": {"perplexity": dense_ppl, "f1": dense_f1},
    "pre_distillation": {"perplexity": ppl_pre["perplexity"], "nll": ppl_pre["nll"],
                          "f1": f1_pre["f1"], "p": f1_pre["precision"], "r": f1_pre["recall"]},
    "post_distillation": {"perplexity": ppl_post["perplexity"], "nll": ppl_post["nll"],
                          "f1": f1_post["f1"], "p": f1_post["precision"], "r": f1_post["recall"]},
    "recovery_pct": {"perplexity": ppl_recovery, "f1": f1_recovery},
}
with open("results/distillation_taylor20.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nresults saved to results/distillation_taylor20.json")

  Distillation recovery — Taylor 20% pruned Qwen 2.5 3B
  Metric                       Dense    Pre-dist   Post-dist     Recovered
------------------------------------------------------------------------
  Perplexity                   1.725       5.789       1.406       107.9%
  F1                           0.614       0.097       0.423        63.1%
  Precision                        ?       0.099       0.632
  Recall                           ?       0.095       0.318

results saved to results/distillation_taylor20.json
